# TCGA-LUAD Phase 7B: GO, Reactome and GSEA
Uses real Phase 6 DESeq2 results bundled in the project ZIP.


In [ ]:
!pip -q install "gseapy>=1.1,<2" pandas numpy matplotlib


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded=files.upload()
archive=next((n for n in uploaded if n.endswith("Phase7B_Project.zip")),None)
assert archive is not None, "Upload TCGA_LUAD_Phase7B_Project.zip"
with zipfile.ZipFile(archive) as z: z.extractall("/content/phase7b")
root=Path("/content/phase7b/TCGA_LUAD_Phase7B")
assert (root/"phase7b.py").exists()


## Prepare the gene ranking and tested-gene background
Check the counts printed by the preflight stage.


In [ ]:
import sys, subprocess
out=Path("/content/phase7b_results")
subprocess.run([sys.executable,str(root/"phase7b.py"),"--dge",str(root/"input_dge.csv"),"--annotations",str(root/"input_annotations.csv"),"--out",str(out)],check=True)


## Run GO Biological Process and Reactome ORA + preranked GSEA
Internet is needed to retrieve the gene-set libraries. This may take several minutes.


In [ ]:
sys.path.insert(0,str(root))
from phase7b import analyze
analyze(out,permutations=1000,seed=42)
print("Enrichment complete:",sorted(p.name for p in out.iterdir()))


## Inspect leading terms


In [ ]:
import pandas as pd
from IPython.display import display,Image
for name in ["ora_GO_BP_up.csv","ora_GO_BP_down.csv","ora_Reactome_up.csv","ora_Reactome_down.csv","gsea_GO_BP.csv","gsea_Reactome.csv"]:
    f=out/name
    if f.exists():
        print("\n",name)
        display(pd.read_csv(f).head(10))
for f in sorted(out.glob("*.png")):
    print(f.name);display(Image(filename=str(f)))


## Export complete outputs


In [ ]:
import shutil
shutil.make_archive("/content/TCGA_LUAD_Phase7B_Results","zip",root_dir=str(out))
files.download("/content/TCGA_LUAD_Phase7B_Results.zip")
